# Cat vs Not-Cat Image Classifier
**Beginner-friendly binary image classification using TensorFlow/Keras**

This notebook walks through every step:
1. Clean corrupted images
2. Split raw data into train / validation folders
3. Load the dataset
4. Visualise sample images
5. Build a simple CNN
6. Train the model
7. Plot accuracy & loss curves
8. Predict on a single new image

## Step 1 — Imports
We bring in everything we'll need up front.

In [ ]:
import os
import shutil
import random
import pathlib

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Reproducibility
random.seed(42)
np.random.seed(42)
tf.random.set_seed(42)

print("TensorFlow version:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices('GPU'))

## Step 2 — Configuration
All tuneable settings live here so you only need to change one place.

In [ ]:
# ── Paths ──────────────────────────────────────────────────────────────────
BASE_DIR        = pathlib.Path(r"D:\VSCODE\cat_classifier\dataset")
SOURCE_CAT      = BASE_DIR / "PetImages" / "Cat"      # raw cat images
SOURCE_DOG      = BASE_DIR / "PetImages" / "Dog"      # raw dog images (= not_cat)
TRAIN_DIR       = BASE_DIR / "train"
VAL_DIR         = BASE_DIR / "validation"

# ── Hyper-parameters ───────────────────────────────────────────────────────
IMAGE_SIZE      = (128, 128)   # resize every image to this
BATCH_SIZE      = 32
EPOCHS          = 5
VAL_SPLIT       = 0.2          # 20 % of images go to validation

# Class names that match the folder names
CLASS_NAMES     = ["cat", "not_cat"]

print("Configuration ready.")
print(f"  Image size : {IMAGE_SIZE}")
print(f"  Batch size : {BATCH_SIZE}")
print(f"  Epochs     : {EPOCHS}")

## Step 3 — Remove Corrupted Images
Some images in the PetImages dataset are broken (0-byte files, bad JPEG headers, etc.).
We use TensorFlow to try to decode each file and skip any that fail.

In [ ]:
def remove_corrupted_images(folder: pathlib.Path) -> int:
    """Delete unreadable image files. Returns how many were removed."""
    removed = 0
    for img_path in folder.glob("*"):
        if img_path.suffix.lower() not in (".jpg", ".jpeg", ".png", ".bmp", ".gif"):
            continue  # skip non-image files
        try:
            raw = img_path.read_bytes()
            tf.image.decode_image(raw, channels=3)   # raises if corrupt
        except Exception:
            img_path.unlink()   # delete the broken file
            removed += 1
    return removed

print("Scanning PetImages/Cat  ...", end=" ")
n_cat = remove_corrupted_images(SOURCE_CAT)
print(f"removed {n_cat} corrupted file(s).")

print("Scanning PetImages/Dog  ...", end=" ")
n_dog = remove_corrupted_images(SOURCE_DOG)
print(f"removed {n_dog} corrupted file(s).")

## Step 4 — Split Dataset into train / validation Folders
We randomly pick 80 % of each class for training and 20 % for validation,
then copy them into the correct sub-folders.

> **Run this cell only once.** If the folders are already populated it will skip.

In [ ]:
def split_and_copy(src_dir: pathlib.Path, class_name: str,
                   train_root: pathlib.Path, val_root: pathlib.Path,
                   val_fraction: float = 0.2) -> None:
    """Copy images from src_dir into train/class_name and val/class_name."""
    train_dest = train_root / class_name
    val_dest   = val_root   / class_name
    train_dest.mkdir(parents=True, exist_ok=True)
    val_dest.mkdir(parents=True, exist_ok=True)

    # Already populated? Skip to avoid duplicating work.
    if any(train_dest.iterdir()):
        print(f"  [{class_name}] train folder already has files — skipping copy.")
        return

    # Gather all images
    images = [f for f in src_dir.glob("*")
              if f.suffix.lower() in (".jpg", ".jpeg", ".png", ".bmp")]
    random.shuffle(images)

    split_idx  = int(len(images) * (1 - val_fraction))
    train_imgs = images[:split_idx]
    val_imgs   = images[split_idx:]

    for img in train_imgs:
        shutil.copy2(img, train_dest / img.name)
    for img in val_imgs:
        shutil.copy2(img, val_dest   / img.name)

    print(f"  [{class_name}] train: {len(train_imgs)}  |  val: {len(val_imgs)}")

print("Splitting dataset ...")
split_and_copy(SOURCE_CAT, "cat",     TRAIN_DIR, VAL_DIR, VAL_SPLIT)
split_and_copy(SOURCE_DOG, "not_cat", TRAIN_DIR, VAL_DIR, VAL_SPLIT)
print("Done.")

## Step 5 — Load the Dataset
`image_dataset_from_directory` reads images from the folder structure, resizes them,
and turns them into batched tensors automatically.

In [ ]:
# ── Training set ───────────────────────────────────────────────────────────
train_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    labels        = "inferred",      # labels come from sub-folder names
    label_mode    = "binary",        # binary → 0 or 1  (2-class problem)
    class_names   = CLASS_NAMES,     # ["cat", "not_cat"]  → cat=0, not_cat=1
    image_size    = IMAGE_SIZE,
    batch_size    = BATCH_SIZE,
    shuffle       = True,
    seed          = 42,
)

# ── Validation set ─────────────────────────────────────────────────────────
val_ds = tf.keras.utils.image_dataset_from_directory(
    VAL_DIR,
    labels        = "inferred",
    label_mode    = "binary",
    class_names   = CLASS_NAMES,
    image_size    = IMAGE_SIZE,
    batch_size    = BATCH_SIZE,
    shuffle       = False,           # no shuffle needed for validation
    seed          = 42,
)

# ── Inspect ────────────────────────────────────────────────────────────────
print("Class names:", CLASS_NAMES)
print("  cat=0, not_cat=1  (binary encoding)")

for images, labels in train_ds.take(1):
    print(f"\nOne training batch:")
    print(f"  images shape : {images.shape}   (batch, height, width, channels)")
    print(f"  labels shape : {labels.shape}")
    print(f"  pixel range  : [{images.numpy().min():.0f}, {images.numpy().max():.0f}]  (raw 0-255)")

## Step 6 — Visualise Sample Images
Always look at your data before training!

In [ ]:
plt.figure(figsize=(12, 8))

# Grab one batch from training
for images, labels in train_ds.take(1):
    for i in range(min(12, BATCH_SIZE)):          # show up to 12 images
        ax = plt.subplot(3, 4, i + 1)
        img = images[i].numpy().astype("uint8")   # convert to uint8 for display
        label_idx = int(labels[i].numpy())
        plt.imshow(img)
        plt.title(CLASS_NAMES[label_idx], fontsize=10)
        plt.axis("off")

plt.suptitle("Sample Training Images", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## Step 7 — Performance Optimisation (Prefetch)
While the GPU processes one batch, the CPU pre-loads the next one in the background.
This makes training noticeably faster.

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
val_ds   = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

print("Dataset pipeline optimised with cache + prefetch.")

## Step 8 — Build the CNN Model

Architecture overview:
```
Input (128×128×3)
  → Rescaling (0-255 → 0-1)          # normalise pixel values
  → Conv2D(32) + ReLU + MaxPool      # detect simple edges/textures
  → Conv2D(64) + ReLU + MaxPool      # detect shapes
  → Conv2D(128) + ReLU + MaxPool     # detect higher-level features
  → Flatten
  → Dense(128) + ReLU
  → Dropout(0.5)                     # reduce overfitting
  → Dense(1) + Sigmoid               # output: probability of being cat
```

The **Rescaling** layer inside the model means you never have to remember to
normalise images manually — it happens automatically.

In [ ]:
model = keras.Sequential([
    # ── Preprocessing ──────────────────────────────────────────────────────
    # Normalise pixel values from [0, 255] to [0.0, 1.0]
    layers.Rescaling(1.0 / 255, input_shape=(*IMAGE_SIZE, 3)),

    # ── Conv Block 1 ───────────────────────────────────────────────────────
    # 32 filters, each 3×3 pixels — learn basic edges and colours
    layers.Conv2D(32, kernel_size=3, activation="relu", padding="same"),
    layers.MaxPooling2D(pool_size=2),    # halve spatial dimensions → 64×64

    # ── Conv Block 2 ───────────────────────────────────────────────────────
    layers.Conv2D(64, kernel_size=3, activation="relu", padding="same"),
    layers.MaxPooling2D(pool_size=2),    # → 32×32

    # ── Conv Block 3 ───────────────────────────────────────────────────────
    layers.Conv2D(128, kernel_size=3, activation="relu", padding="same"),
    layers.MaxPooling2D(pool_size=2),    # → 16×16

    # ── Classifier head ────────────────────────────────────────────────────
    layers.Flatten(),                    # 16×16×128 = 32768 values → 1-D
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.5),                 # randomly zero out 50 % of neurons
                                         # during training to reduce overfitting
    layers.Dense(1, activation="sigmoid"),  # output 0-1 probability
], name="cat_classifier")

model.summary()

## Step 9 — Compile the Model
- **Optimizer**: Adam — adaptively adjusts learning rates, works well out of the box.
- **Loss**: Binary cross-entropy — the standard loss for 2-class classification.
- **Metric**: Accuracy — easy to interpret: percentage of correct predictions.

In [ ]:
model.compile(
    optimizer = "adam",
    loss      = "binary_crossentropy",
    metrics   = ["accuracy"],
)

print("Model compiled.")

## Step 10 — Train the Model
We train for 5 epochs. Each epoch = one full pass over all training images.
Validation metrics are computed at the end of every epoch so you can see
how well the model generalises to images it has never seen.

In [ ]:
history = model.fit(
    train_ds,
    epochs            = EPOCHS,
    validation_data   = val_ds,
    verbose           = 1,
)

## Step 11 — Plot Training & Validation Curves
- **Accuracy** should rise for both train and validation.
- **Loss** should fall for both.
- A large gap between train and validation lines → overfitting (model memorised training data).

In [ ]:
acc      = history.history["accuracy"]
val_acc  = history.history["val_accuracy"]
loss     = history.history["loss"]
val_loss = history.history["val_loss"]
epochs_range = range(1, EPOCHS + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# ── Accuracy ───────────────────────────────────────────────────────────────
ax1.plot(epochs_range, acc,     label="Train accuracy",      marker="o")
ax1.plot(epochs_range, val_acc, label="Validation accuracy", marker="o")
ax1.set_title("Accuracy over epochs")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Accuracy")
ax1.legend()
ax1.set_xticks(epochs_range)
ax1.set_ylim(0, 1)

# ── Loss ───────────────────────────────────────────────────────────────────
ax2.plot(epochs_range, loss,     label="Train loss",      marker="o")
ax2.plot(epochs_range, val_loss, label="Validation loss", marker="o")
ax2.set_title("Loss over epochs")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Loss")
ax2.legend()
ax2.set_xticks(epochs_range)

plt.suptitle("Training History", fontsize=14)
plt.tight_layout()
plt.savefig(r"D:\VSCODE\cat_classifier\training_history.png", dpi=120)
plt.show()
print("Plot saved to training_history.png")

## Step 12 — Save the Trained Model
Save the model so you can reload it later without retraining.

In [ ]:
MODEL_PATH = r"D:\VSCODE\cat_classifier\cat_classifier.keras"
model.save(MODEL_PATH)
print(f"Model saved to: {MODEL_PATH}")

## Step 13 — Predict on a Single New Image
Change `TEST_IMAGE_PATH` to any image on your computer to classify it.

In [ ]:
# ── Point this at any image you want to test ───────────────────────────────
# Example: use the first image found in the validation/cat folder
sample_dir   = pathlib.Path(r"D:\VSCODE\cat_classifier\dataset\validation\cat")
TEST_IMAGE_PATH = next(sample_dir.glob("*.jpg"), None)   # grab first .jpg

if TEST_IMAGE_PATH is None:
    print("No test image found — set TEST_IMAGE_PATH manually.")
else:
    # ── Load & preprocess ──────────────────────────────────────────────────
    raw   = tf.io.read_file(str(TEST_IMAGE_PATH))
    img   = tf.image.decode_image(raw, channels=3, expand_animations=False)
    img   = tf.image.resize(img, IMAGE_SIZE)           # resize to 128×128
    img   = tf.expand_dims(img, axis=0)                # add batch dimension → (1, 128, 128, 3)
    # Note: we do NOT divide by 255 here because the Rescaling layer
    #       inside the model does that automatically.

    # ── Predict ────────────────────────────────────────────────────────────
    pred_prob = model.predict(img, verbose=0)[0][0]    # sigmoid output 0-1
    pred_class = CLASS_NAMES[int(pred_prob > 0.5)]     # >0.5 → not_cat, else cat

    # ── Display ────────────────────────────────────────────────────────────
    display_img = tf.squeeze(img).numpy().astype("uint8")
    plt.figure(figsize=(4, 4))
    plt.imshow(display_img)
    plt.axis("off")
    plt.title(
        f"Prediction: {pred_class}\n"
        f"(cat prob: {1 - pred_prob:.2%}  |  not_cat prob: {pred_prob:.2%})",
        fontsize=11
    )
    plt.tight_layout()
    plt.show()

    print(f"Image       : {TEST_IMAGE_PATH.name}")
    print(f"Prediction  : {pred_class}")
    print(f"Confidence  : cat={1-pred_prob:.2%}  not_cat={pred_prob:.2%}")

---
## Reopen & Predict (No Retraining Needed)

After the model has been trained and saved once, you **only need to run the two cells below** on any future session.
Skip all the training steps — just load the saved model and predict."

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# QUICK START — run this cell after reopening VS Code (no retraining needed)
# ══════════════════════════════════════════════════════════════════════════════

import pathlib
import tensorflow as tf
import matplotlib.pyplot as plt

# ── Settings (must match what you used during training) ────────────────────
IMAGE_SIZE  = (128, 128)
CLASS_NAMES = ["cat", "not_cat"]   # cat=0 (prob < 0.5), not_cat=1 (prob >= 0.5)
MODEL_PATH  = r"D:\VSCODE\cat_classifier\cat_classifier.keras"

# ── Load the saved model ───────────────────────────────────────────────────
model = tf.keras.models.load_model(MODEL_PATH)
print(f"Model loaded from: {MODEL_PATH}")
model.summary()

In [ ]:
# ── Change this path to any image you want to classify ─────────────────────
TEST_IMAGE_PATH = r"D:\VSCODE\cat_classifier\dataset\validation\cat"   # folder
# Or point directly at one file, e.g.:
# TEST_IMAGE_PATH = r"C:\Users\YourName\Pictures\my_cat.jpg"

# ── Auto-pick first image if a folder was given ────────────────────────────
p = pathlib.Path(TEST_IMAGE_PATH)
if p.is_dir():
    p = next(p.glob("*.jpg"), next(p.glob("*.png"), None))

if p is None or not p.exists():
    print("Image not found — set TEST_IMAGE_PATH to a valid image file.")
else:
    # ── Preprocess ─────────────────────────────────────────────────────────
    raw        = tf.io.read_file(str(p))
    img_tensor = tf.image.decode_image(raw, channels=3, expand_animations=False)
    img_tensor = tf.image.resize(img_tensor, IMAGE_SIZE)   # → (128, 128, 3)
    img_batch  = tf.expand_dims(img_tensor, axis=0)        # → (1, 128, 128, 3)
    # The Rescaling layer inside the model handles 0-255 → 0-1 automatically.

    # ── Predict ────────────────────────────────────────────────────────────
    pred_prob  = model.predict(img_batch, verbose=0)[0][0]   # 0.0 – 1.0
    pred_class = CLASS_NAMES[int(pred_prob >= 0.5)]          # threshold at 0.5
    cat_conf   = 1.0 - pred_prob                             # lower prob = more cat

    # ── Display ────────────────────────────────────────────────────────────
    plt.figure(figsize=(4, 4))
    plt.imshow(img_tensor.numpy().astype("uint8"))
    plt.axis("off")
    plt.title(
        f"Prediction : {pred_class}\n"
        f"cat {cat_conf:.1%}  |  not_cat {pred_prob:.1%}",
        fontsize=12
    )
    plt.tight_layout()
    plt.show()

    print(f"File       : {p.name}")
    print(f"Prediction : {pred_class}")
    print(f"cat        : {cat_conf:.2%}")
    print(f"not_cat    : {pred_prob:.2%}")